# Day 1 extra exercise: summarize any website, including JavaScript-rendered ones

`week1/day1.ipynb` scrapes with `requests` + BeautifulSoup (`scraper.py`), which only sees the HTML the server sends. Sites that build the page with JavaScript can send a nearly empty shell, leaving little or nothing to summarize. The quotes sandbox below is a clear example.

[`js_scraper.py`](js_scraper.py) is a drop-in replacement for `scraper.py` (same `fetch_website_contents` and `fetch_website_links` functions) that:

1. tries the fast static fetch first,
2. detects a JavaScript shell (almost no text, an empty `#root` / `#__next` mount point, "please enable JavaScript", or a 403/429/503 block),
3. only then renders the page in headless Chromium with **Playwright**: waits for the network to settle, clicks common cookie "Accept" buttons, scrolls to load lazy content, and also reads text inside shadow DOM (web components),
4. strips navigation, headers, footers and cookie banners, so more of the character budget goes to real content.

Everything else follows day 1: same prompts, same `messages_for`, `summarize` and `display_summary`.

## Setup (once)

Installs Playwright and the headless Chromium it drives (~150 MB). The course's `.venv` is created by `uv` and may have no `pip`, so this falls back to `uv pip`.

In [ ]:
import shutil, subprocess, sys

if subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True).returncode == 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"], check=True)
elif shutil.which("uv"):
    subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, "playwright"], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
print("Playwright and Chromium are ready")

In [ ]:
# imports

import os
from dotenv import load_dotenv
from js_scraper import fetch_website_contents, fetch_website_links, fetch_page
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
# Load environment variables in a file called .env (same as day 1)

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in week1 to identify & fix!")
elif api_key.strip() != api_key:
    print("An API key was found, but it has spaces or tabs at the start or end - please remove them")
else:
    print("API key found and looks good so far!")

openai = OpenAI()
MODEL = "gpt-6-luna"

## Static vs rendered: see the difference

[quotes.toscrape.com/js](https://quotes.toscrape.com/js/) is a public scraping sandbox whose quotes are inserted by JavaScript. `mode="static"` uses `requests` only, like `scraper.py`, but with stricter text cleaning; the default `mode="auto"` notices the empty page and switches to the browser. No API call is made here.

In [ ]:
JS_URL = "https://quotes.toscrape.com/js/"

static = fetch_page(JS_URL, mode="static")
print(f"STATIC ({len(static.text)} characters):\n{static.text}\n")

rendered = fetch_page(JS_URL, verbose=True)
print(f"\nRENDERED ({len(rendered.text)} characters):\n{rendered.text[:600]}")

## The day 1 summarizer, unchanged except for the scraper

In [ ]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


def summarize(url, mode="auto"):
    website = fetch_website_contents(url, mode=mode)
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=messages_for(website)
    )
    return response.choices[0].message.content


def display_summary(url, mode="auto"):
    summary = summarize(url, mode)
    display(Markdown(summary))

In [ ]:
# A static site: no browser is launched, so this is as fast as day 1
display_summary("https://edwarddonner.com")

In [ ]:
# The example from the day 1 exercise. Results vary: the simple scraper returned real text from it when tested
display_summary("https://openai.com")

In [ ]:
# A page whose content exists only after JavaScript runs
display_summary("https://quotes.toscrape.com/js/")

## Try more

- `display_summary(url, mode="browser")` forces rendering; `mode="static"` never launches a browser.
- `fetch_page(url, verbose=True)` shows which path was used and why.
- `fetch_website_links(url)` also returns JavaScript-generated links, which helps with the day 5 brochure exercise.

Sites that need a login or show a CAPTCHA (Reddit, for example) will still return only their "prove you're human" page. Please respect each site's terms of service.

In [ ]:
display_summary("https://www.airbnb.com")